# Train — DBpedia (in-batch negatives)

**Purpose:** Trains `HybridSPARQLEncoder` on DBpedia semantic triplets using **in-batch negatives** (InfoNCE): one positive per anchor, with the other batch items acting as negatives.

**Input:** `../../data/jaccard_triplets_dbpedia.json` — unzip `data/jaccard_triplets_dbpedia.zip` once before running.

**Splits:** `../../data/splits_dbpedia.json` (seed 42 — train 3770 / valid 471 / test 472). Set `REGENERATE_SPLITS = True` to redo.

**Output:** `../../checkpoints/dbpedia_model_semantic-20.pt` — trained model weights.

**Config:** see `sparqlce.config.Config` (device, epochs, learning rate, temperature). Batch size for in-batch training is 16.

**Note:** As of the canonicalization refactor, `SPARQLGraphBuilder` automatically canonicalizes variables and drops no-op `FILTER(true)`. To additionally train with random variable-rename augmentation, pass `augment_rename_prob=0.3` to `SemanticSPARQLDataset(...)`.

**Sections:** 1. Imports and data load | 2. Splits | 3. Dataset and DataLoaders (in-batch) | 4. Loss | 5. Training and save.


### 1. Imports and data load

In [ ]:
import sys, os

# Add project root (two levels up from notebooks/training/) so `sparqlce` is importable.
PROJECT_ROOT = os.path.abspath("../..")
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import json, random, numpy as np, torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset

from sparqlce.config import Config
from sparqlce.graph_builder import SPARQLGraphBuilder
from sparqlce.encoder import HybridSPARQLEncoder
from sparqlce.dataset import SemanticSPARQLDataset, semantic_collate_fn_inbatch

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Input path: unzip data/jaccard_triplets_dbpedia.zip into data/ before running.
SEMANTIC_PATH = "../../data/jaccard_triplets_dbpedia.json"
if not os.path.exists(SEMANTIC_PATH):
    raise FileNotFoundError(
        f"{SEMANTIC_PATH} not found. From the repo root, run:\n"
        "    cd data && unzip jaccard_triplets_dbpedia.zip && cd .."
    )

with open(SEMANTIC_PATH, "r", encoding="utf-8") as f:
    raw_sem = json.load(f)

print(f"Total semantic entries: {len(raw_sem)}")


### 2. Splits

In [ ]:
import os

# Reuse the committed splits file (seed=42, 3770/471/472). Set to True to regenerate.
SPLITS_PATH = "../../data/splits_dbpedia.json"
REGENERATE_SPLITS = False

load_existing = os.path.exists(SPLITS_PATH) and not REGENERATE_SPLITS
if load_existing:
    with open(SPLITS_PATH, "r", encoding="utf-8") as f:
        split_info = json.load(f)
    train_idx = split_info["train_idx"]
    valid_idx = split_info["valid_idx"]
    test_idx = split_info["test_idx"]
    n_total = len(train_idx) + len(valid_idx) + len(test_idx)
    if n_total != len(raw_sem):
        load_existing = False
        print(f"Splits stale (dataset has {len(raw_sem)} items, splits have {n_total}). Regenerating...")

if load_existing:
    print(f"Loaded existing splits from {SPLITS_PATH}")
    print(f"Train: {len(train_idx)}, Valid: {len(valid_idx)}, Test: {len(test_idx)}")
else:
    indices = list(range(len(raw_sem)))
    random.shuffle(indices)

    n = len(indices)
    train_size = int(0.8 * n)
    valid_size = int(0.1 * n)
    test_size  = n - train_size - valid_size

    train_idx = indices[:train_size]
    valid_idx = indices[train_size : train_size + valid_size]
    test_idx  = indices[train_size + valid_size :]

    print(f"Train: {len(train_idx)}, Valid: {len(valid_idx)}, Test: {len(test_idx)}")

    os.makedirs(os.path.dirname(SPLITS_PATH), exist_ok=True)
    split_info = {
        "seed": SEED,
        "train_idx": train_idx,
        "valid_idx": valid_idx,
        "test_idx": test_idx,
    }
    with open(SPLITS_PATH, "w", encoding="utf-8") as f:
        json.dump(split_info, f, indent=2)

    print(f"Saved splits to {SPLITS_PATH}")


### Dataset and DataLoaders (in-batch)

In [ ]:
print("Initializing semantic dataset...")
builder = SPARQLGraphBuilder()

# To train with random variable-rename augmentation, set augment_rename_prob > 0
# (e.g. 0.3). The graph builder already canonicalizes variables and skips
# FILTER(true) for every input, so augmentation is purely additional robustness.
full_sem = SemanticSPARQLDataset(
    SEMANTIC_PATH,
    builder,
    in_batch_negatives=True,
    augment_rename_prob=0.0,
    augment_seed=SEED,
)
BATCH_SIZE_TRAIN = 16

print(f"Total semantic anchors: {len(full_sem)}")

train_ds = Subset(full_sem, train_idx)
valid_ds = Subset(full_sem, valid_idx)

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE_TRAIN,
    collate_fn=semantic_collate_fn_inbatch,
    shuffle=True,
    num_workers=0,
)
valid_loader = DataLoader(
    valid_ds,
    batch_size=BATCH_SIZE_TRAIN,
    collate_fn=semantic_collate_fn_inbatch,
    shuffle=False,
    num_workers=0,
)

print(f"Train: {len(train_ds)} | Valid: {len(valid_ds)}")


### Loss (in-batch InfoNCE)

In [ ]:
import torch


def infonce_inbatch(anchor_emb, pos_emb, temperature: float):
    anchor_emb = F.normalize(anchor_emb, dim=1)
    pos_emb = F.normalize(pos_emb, dim=1)
    logits = (anchor_emb @ pos_emb.T) / temperature
    labels = torch.arange(logits.size(0), device=logits.device)
    return F.cross_entropy(logits, labels)


### Training loop and save

In [ ]:
model = HybridSPARQLEncoder().to(Config.DEVICE)
model.train()
optimizer = torch.optim.AdamW(model.parameters(), lr=Config.LEARNING_RATE)

print(f"Training on {Config.DEVICE} with {len(train_ds)} samples...")

for epoch in range(Config.EPOCHS):
    total_loss = 0.0

    for anchor, pos, pos_js in train_loader:
        anchor = anchor.to(Config.DEVICE)
        pos = pos.to(Config.DEVICE)
        pos_js = pos_js.to(Config.DEVICE)

        optimizer.zero_grad()
        a_emb = model(anchor)
        p_emb = model(pos)
        loss_ce = infonce_inbatch(a_emb, p_emb, Config.TEMPERATURE)
        loss_reg = ((a_emb * p_emb).sum(dim=1) - pos_js).pow(2).mean()
        loss = loss_ce + 0.3 * loss_reg

        loss.backward()
        optimizer.step()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        total_loss += loss.item()

    avg_train_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch+1}/{Config.EPOCHS} | Train Loss: {avg_train_loss:.4f}")

    model.eval()
    with torch.no_grad():
        sims_pos, count = 0.0, 0
        for batch in valid_loader:
            anchor, pos, pos_js = batch
            anchor = anchor.to(Config.DEVICE)
            pos = pos.to(Config.DEVICE)
            a_emb = F.normalize(model(anchor), dim=1)
            p_emb = F.normalize(model(pos), dim=1)
            sims_pos += (a_emb * p_emb).sum(dim=1).sum().item()
            count += anchor.size(0)
        print(f"          Valid sim(anchor,pos): {sims_pos / count:.4f}")

    model.train()


In [ ]:
CKPT_DIR = "../../checkpoints"
CKPT_PATH = os.path.join(CKPT_DIR, "dbpedia_model_semantic-20.pt")
os.makedirs(CKPT_DIR, exist_ok=True)
torch.save(model.state_dict(), CKPT_PATH)
print(f"Semantic training complete. Model saved to {CKPT_PATH}")
